# EEG Preprocessing Pipeline (Adapted from Nieto et al.)

This notebook adapts the official preprocessing pipeline released with the
**Thinking Out Loud** inner-speech EEG dataset (Nieto et al., 2022).

**Source:** N. Nieto et al., *"Thinking out loud, an open-access EEG-based BCI
dataset for inner speech recognition,"* Scientific Data, 2022.
Original code: https://github.com/N-Nieto/Inner_Speech_Dataset

**Output Preprocessing:** filtered, re-referenced, and epoched EEG data
(128 channels, per subject/session) 


# Channel Selection (29 Channels) and Inner Speech Action Interval Selection

This notebook takes the preprocessed data (128-channel BioSemi montage) and:

1. Maps 29 standard 10-20 channels to their nearest BioSemi-128 electrodes.
2. reduces the data to these 29 channels.
3. Crops each trial to the 2.5 s inner-speech action interval.
4. Saves the final `X`, `y` arrays 

## 1 — Mapping 29 Standard 10-20 Channels to BioSemi-128 Electrodes

For each of the 29 target 10-20 channels, the nearest BioSemi-128 electrode is
found by Euclidean distance between the two montages' 3D sensor positions
(printed in the table below and saved to
`mapping_29_to_biosemi128_by_coordinates.csv`).

This automatic mapping was then manually cross-checked against the official
BioSemi-128 electrode layout reference. For 6 of the 29 channels
(**T7, T8, P4, O1, O2, Oz**), the nearest-neighbor result did not correspond
to the anatomically expected electrode, so the mapping was manually corrected;
the final, corrected assignment used throughout this study is the
`mapping_29` dictionary defined in Section 2 below, which therefore differs
from the raw distance-based table for these 6 channels.


In [1]:
import mne
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

# 29 selected channels
target_1020_channels = [
    "Fp1", "Fp2", "Fpz", "F3", "F4", "F7", "F8", "Fz",
    "FC3", "FC4", "FT7", "FT8", "FCz",
    "C3", "C4", "Cz", "T7", "T8",
    "CP3", "CP4", "CPz", "TP7", "TP8",
    "P3", "P4", "Pz", "O1", "O2", "Oz"
]

# Montage BioSemi 128 channels 
biosemi_montage = mne.channels.make_standard_montage("biosemi128")

# Montage  10-20 / 10-10 Standard
standard_montage = mne.channels.make_standard_montage("standard_1020")

biosemi_pos = biosemi_montage.get_positions()["ch_pos"]
standard_pos = standard_montage.get_positions()["ch_pos"]
biosemi_channels = [
    ch for ch in biosemi_montage.ch_names
    if ch.startswith(("A", "B", "C", "D"))
]
biosemi_xyz = np.array([biosemi_pos[ch] for ch in biosemi_channels])
rows = []

for target_ch in target_1020_channels:
    if target_ch not in standard_pos:
        print(f"Warning: {target_ch} not found in standard_1020 montage")
        continue

    target_xyz = np.array(standard_pos[target_ch]).reshape(1, -1)

    distances = cdist(target_xyz, biosemi_xyz)[0]

    nearest_idx = np.argmin(distances)
    nearest_biosemi_ch = biosemi_channels[nearest_idx]
    nearest_distance = distances[nearest_idx]

    rows.append({
        "target_10_20": target_ch,
        "nearest_biosemi_128": nearest_biosemi_ch,
        "distance": nearest_distance
    })

mapping_df = pd.DataFrame(rows)

print(mapping_df)

mapping_df.to_csv("mapping_29_to_biosemi128_by_coordinates.csv", index=False)


   target_10_20 nearest_biosemi_128  distance
0           Fp1                 C29  0.007362
1           Fp2                 C16  0.006604
2           Fpz                 C17  0.006885
3            F3                  D4  0.011778
4            F4                  C4  0.011991
5            F7                  D7  0.016919
6            F8                  C7  0.014805
7            Fz                 C21  0.009841
8           FC3                 D12  0.011296
9           FC4                 B31  0.010585
10          FT7                  D8  0.019580
11          FT8                 B27  0.018165
12          FCz                 C23  0.009813
13           C3                 D19  0.012118
14           C4                 B22  0.011230
15           Cz                  A1  0.010569
16           T7                 D24  0.015856
17           T8                 B14  0.016442
18          CP3                  A6  0.015295
19          CP4                  B3  0.018372
20          CPz                  A

## 2 — Channel Selection On Data

final shape: `trials x 29 channels x samples`


**Input:** This step reads the per-subject, per-session epoch files produced
by the Nieto-adapted preprocessing pipeline described above. That pipeline
defines `save_dir` (the preprocessed-data output directory) and writes one
`*_eeg-epo.fif` file and one `*_events.dat` file per subject and session,
following this naming convention:

```
save_dir/
└── sub-<ID>/
    └── ses-0<N>/
        ├── sub-<ID>_ses-0<N>_eeg-epo.fif
        └── sub-<ID>_ses-0<N>_events.dat
```

**Note — this notebook is not standalone-runnable.** The preprocessing step
itself (which produces the files above) is not reproduced here, since it is
unmodified third-party code — see the citation and repository link at the
top of this notebook. To run the cell below, first execute Nieto et al.'s
original preprocessing pipeline so that `save_dir` points to a populated
output directory in the structure shown above.


In [ ]:
base_path = save_dir / "sub-10"
sessions = ["ses-01", "ses-02", "ses-03"]

# 29-channel mapping after the mannual check 
mapping_29 = {
    "Fp1": "C29", "Fp2": "C16", "Fpz": "C17",
    "F3": "D4",   "F4": "C4",   "F7": "D7",   "F8": "C7",   "Fz": "C21",
    "FC3": "D12", "FC4": "B31", "FT7": "D8",  "FT8": "B27", "FCz": "C23",
    "C3": "D19",  "C4": "B22",  "Cz": "A1",
    "T7": "D23",  "T8": "B26",
    "CP3": "A6",  "CP4": "B3",  "CPz": "A3",
    "TP7": "D31", "TP8": "B11",
    "P3": "A18",  "P4": "A31",  "Pz": "A19",
    "O1": "A15",  "O2": "A28",  "Oz": "A23"
}

selected_29_channels = list(mapping_29.values())

all_X = []
all_y = []

for ses in sessions:

    print(f"\n{'='*60}")
    print(f"Processing {ses}")
    print(f"{'='*60}")

    # load events file
    events_df = pd.read_csv(
        base_path / ses / f"sub-10_{ses}_events.dat",
        index_col=0
    )

    # load epochs
    epochs = mne.read_epochs(
        base_path / ses / f"sub-10_{ses}_eeg-epo.fif",
        preload=True
    )

    # -------------------------------------------------
    # 1. select inner speech trials
    # -------------------------------------------------
    inner_positions = np.where(events_df["condition"].values == 1)[0]

    epochs_inner = epochs[inner_positions]

    # -------------------------------------------------
    # 2. keep only 29 mapped channels
    # -------------------------------------------------
    missing = [ch for ch in selected_29_channels if ch not in epochs_inner.ch_names]

    if missing:
        raise ValueError(f"Missing channels in {ses}: {missing}")

    epochs_29 = epochs_inner.copy().pick_channels(
        selected_29_channels,
        ordered=True
    )

    epochs_29.reorder_channels(selected_29_channels)

    # -------------------------------------------------
    # 3. extract X
    # -------------------------------------------------
    X_session = epochs_29.get_data()  # (trials, 29, samples)

    # -------------------------------------------------
    # 4. extract y
    # -------------------------------------------------
    y_session = epochs_29.events[:, 2]

    print("Session shape:", X_session.shape)
    print("Labels:", np.unique(y_session, return_counts=True))

    all_X.append(X_session)
    all_y.append(y_session)


# -------------------------------------------------
# 5. concatenate sessions
# -------------------------------------------------

X_29channels = np.concatenate(all_X, axis=0)
y_29channels = np.concatenate(all_y, axis=0)

print(f"\n{'='*60}")
print("FINAL DATASET")
print(f"{'='*60}")

print("X shape:", X_29channels.shape)
print("y shape:", y_29channels.shape)

print("Trials   :", X_29channels.shape[0])
print("Channels :", X_29channels.shape[1])
print("Samples  :", X_29channels.shape[2])

print("Labels distribution:")
print(np.unique(y_29channels, return_counts=True))


## 3 — Cropping the 2.5s Inner-Speech Action interval

The full epoch spans -0.5 s to 4.0 s (concentration interval, cue interval, action interval, relax interval  ). Only the 2.5 s
action interval (1 s to 3.5 s), during which subjects perform the inner-speech
task, is kept for further analysis.


In [ ]:
n_times = X_29channels.shape[2]
times = np.linspace(-0.5, 4.0, n_times)
time_mask = (times >= 0.5) & (times <= 3)
X_InnerSpeech_Task = X_29channels[:, :, time_mask]
print(X_InnerSpeech_Task.shape)
y_InnerSpeech_Task = y_29channels
print(y_InnerSpeech_Task.shape)

## 4 — Save Final Preprocessed Arrays


In [ ]:
import os
save_path_y = "./data/processed/y10_InnerSpeech_Task.npy"
os.makedirs(os.path.dirname(save_path_y), exist_ok=True)
np.save(save_path_y, y_InnerSpeech_Task)
save_path_x = "./data/processed/X10_InnerSpeech_Task.npy"
os.makedirs(os.path.dirname(save_path_x), exist_ok=True)
np.save(save_path_x, X_InnerSpeech_Task)
